In [17]:
!pip -q install langchain langchain-core langchain-community transformers accelerate torch


In [18]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [30]:
from transformers import pipeline

llm = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",   # Fast, ~0.5B model
    max_new_tokens=128,
    temperature=0.3,
    clean_up_tokenization_spaces=False,
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [31]:
messages = [
    {"role": "user", "content": "Say hello in one sentence."}
]

response = llm(messages)

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello! It's always nice to meet you!


In [23]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template
prompt = PromptTemplate.from_template(
    "Write a {adjective} joke about {content}."
)

# Fill the variables
formatted_prompt = prompt.invoke({
    "adjective": "funny",
    "content": "AI"
})

print(formatted_prompt)

text='Write a funny joke about AI.'


In [24]:
response = llm([
    {"role": "user", "content": formatted_prompt.text}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why don't AI jokes ever get told at parties?

Because they're always too "code" for the audience!


In [25]:
zero_shot_prompt = """
Classify the sentiment of the following review as Positive, Negative, or Neutral.

Review: "The food was delicious and the service was excellent."

Sentiment:
"""

response = llm([
    {"role": "user", "content": zero_shot_prompt}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Neutral


In [26]:
one_shot_prompt = """
Review: "The movie was amazing!"
Sentiment: Positive

Review: "The product arrived damaged and late."
Sentiment:
"""

response = llm([
    {"role": "user", "content": one_shot_prompt}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: Negative


In [27]:
few_shot_prompt = """
Review: "The movie was amazing!"
Sentiment: Positive

Review: "The food was terrible."
Sentiment: Negative

Review: "The package arrived on time."
Sentiment: Positive

Review: "The customer support was unhelpful and slow."
Sentiment:
"""

response = llm([
    {"role": "user", "content": few_shot_prompt}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: Negative


In [32]:
cot_prompt = """
Solve this step by step.

A store has 25 apples. It sells 8 apples and then receives 12 more apples.
How many apples does the store have now?
"""

response = llm([
    {"role": "user", "content": cot_prompt}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure! Let's solve this problem step by step:

1. **Initial number of apples**: The store starts with 25 apples.

2. **Apples sold**: The store sells 8 apples. So, we subtract 8 from the initial amount:
   \[
   25 - 8 = 17
   \]
   Now, the store has 17 apples left.

3. **Apples received**: The store receives 12 more apples. So, we add these to the current total:
   \[
   17 + 12 = 29
   \]




In [34]:
direct_prompt = """
Answer with ONLY the final answer. Do not show steps or explanation.

A store has 25 apples. It sells 8 apples and then receives 12 more apples.
How many apples does the store have?
"""

response = llm([
    {"role": "user", "content": direct_prompt}
])

print(response[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The store initially has 25 apples. After selling 8 apples, it has \(25 - 8 = 17\) apples left. Then, it receives 12 more apples, so the total number of apples in the store now is \(17 + 12 = 29\). Therefore, the store has 29 apples.


In [35]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "Explain {topic} in simple words.")
])

formatted_messages = chat_prompt.invoke({
    "topic": "Machine Learning"
})

print(formatted_messages)

messages=[SystemMessage(content='You are a helpful AI assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Machine Learning in simple words.', additional_kwargs={}, response_metadata={})]


In [36]:
summary_prompt = PromptTemplate.from_template(
    """
    Summarize the following text in 3 bullet points.

    Text:
    {text}
    """
)

formatted_prompt = summary_prompt.invoke({
    "text": "Artificial Intelligence enables computers to perform tasks that usually require human intelligence, including learning, reasoning, and decision-making."
})

print(formatted_prompt.text)


    Summarize the following text in 3 bullet points.

    Text:
    Artificial Intelligence enables computers to perform tasks that usually require human intelligence, including learning, reasoning, and decision-making.
    


In [37]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

sample_text = "This is a sample LLM response."
print(output_parser.invoke(sample_text))

This is a sample LLM response.


In [38]:
from langchain_core.runnables import RunnableLambda

def format_prompt(inputs):
    return f"Write a {inputs['adjective']} joke about {inputs['content']}."

formatter = RunnableLambda(format_prompt)

print(formatter.invoke({
    "adjective": "funny",
    "content": "AI"
}))

Write a funny joke about AI.


In [39]:
prompt = PromptTemplate.from_template(
    "Write a {adjective} joke about {content}."
)

chain = prompt | RunnableLambda(lambda x: x.to_string())

In [40]:
formatted = chain.invoke({
    "adjective": "funny",
    "content": "AI"
})

print(formatted)

Write a funny joke about AI.


In [41]:
def local_llm(prompt_text):
    response = llm([
        {"role": "user", "content": prompt_text}
    ])
    return response[0]["generated_text"][-1]["content"]

llm_runnable = RunnableLambda(local_llm)

full_chain = (
    PromptTemplate.from_template("Write a {adjective} joke about {content}.")
    | RunnableLambda(lambda x: x.to_string())
    | llm_runnable
    | StrOutputParser()
)

In [42]:
result = full_chain.invoke({
    "adjective": "funny",
    "content": "AI"
})

print(result)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the AI go to therapy? Because it was feeling lonely!


In [43]:
parallel_chain = RunnableParallel(
    summary=RunnableLambda(lambda x: local_llm(f"Summarize: {x}")),
    translation=RunnableLambda(lambda x: local_llm(f"Translate to French: {x}")),
    sentiment=RunnableLambda(lambda x: local_llm(f"Classify sentiment: {x}")),
)

In [44]:
result = parallel_chain.invoke(
    "The movie was amazing and I loved every minute of it."
)

print(result)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'summary': "I'm sorry, but as an AI language model, I don't have personal experiences or emotions like humans do. However, I can provide you with some general information about the movie based on your description.\n\nThe movie you mentioned is likely a film or TV show that you enjoyed watching. Without more context, it's difficult to give a specific summary, but generally, movies are characterized by their plot, characters, cinematography, and overall quality. If you could provide me with more details about the movie, such as its title, genre, director, actors, or any other relevant information, I'd be happy to help summarize it for you", 'translation': 'Voici la traduction en français :\n\nLe film était merveilleux et j\'adore chaque seconde de lui.\n\nCette phrase exprime l\'amour profond que l\'on ressent pour un film, tout en soulignant son excellentement. "Merveilleux" est utilisé pour exprimer le sentiment d\'être enthousiaste ou ravi, tandis que "j\'adore chaque seconde de lui"

In [45]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

parallel_chain = RunnableParallel(
    summary=RunnableLambda(
        lambda x: local_llm(f"Summarize in ONE sentence only:\n{x}")
    ),
    translation=RunnableLambda(
        lambda x: local_llm(f"Translate ONLY to French. Do not explain:\n{x}")
    ),
    sentiment=RunnableLambda(
        lambda x: local_llm(f"Reply with ONLY Positive, Negative, or Neutral.\n{x}")
    ),
)

result = parallel_chain.invoke(
    "The movie was amazing and I loved every minute of it."
)

print(result)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'summary': '"The movie was outstanding and left me thoroughly impressed."', 'translation': "Le film était merveilleux et j'aime particulièrement chaque seconde de lui.", 'sentiment': 'Positive'}


In [46]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    "uppercase": lambda x: x.upper(),
    "length": lambda x: len(x),
    "reverse": lambda x: x[::-1],
})

result = parallel_chain.invoke("langchain")

print(result)

{'uppercase': 'LANGCHAIN', 'length': 9, 'reverse': 'niahcgnal'}
